Importing everything

In [28]:
import os
import sys

REPO_ROOT = "/Users/mauraschorr/Downloads/recommender-system"
SRC_PATH = os.path.join(REPO_ROOT, "src")

print("Notebook folder:", os.getcwd())
print("Repo exists:", os.path.exists(REPO_ROOT))
print("Src exists:", os.path.exists(SRC_PATH))
print("Files in src:", os.listdir(SRC_PATH))

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print("First sys.path entry:", sys.path[0])


Notebook folder: /Users/mauraschorr/Downloads/recommender-system/notebook
Repo exists: True
Src exists: True
Files in src: ['temp', 'svd_rec.py', '__pycache__', 'nn_rec.py', 'data_utils.py', 'random_rec.py']
First sys.path entry: /Users/mauraschorr/Downloads/recommender-system/src


In [29]:
from data_utils import load_ratings, build_folds
from surprise import Dataset, Reader
import pandas as pd 

ratings_path = "/Users/mauraschorr/Downloads/ml-latest-small/ratings.csv"

ratings = pd.read_csv(ratings_path)

#convert dataset to surprise format 
reader = Reader(rating_scale=(1, 5))

ratings_data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

#build folds consistent with other datasets 
folds = build_folds(ratings_data)



kNN implementation 

In [32]:
#kNN 
import time
from surprise import KNNBasic, accuracy

#defining knn model 
def run_knn_cv(folds, k=80, user_based=False, save_predictions=False):
    fold_results = []
    all_predictions = []

    for fold_num, (trainset, testset) in enumerate(folds, start=1):
        sim_options = {
            "name": "cosine",
            "user_based": user_based #it compares movies to movies not users to users 
        }

        model = KNNBasic(
            k=k,
            min_k=1,
            sim_options=sim_options,
            verbose=False
        )

        #track the time it runs on each fold 
        start_time = time.time()

        model.fit(trainset)
        predictions = model.test(testset)

        end_time = time.time()
        elapsed_time = end_time - start_time

        #stats for the run on each fold 
        rmse = accuracy.rmse(predictions, verbose=False)
        mae = accuracy.mae(predictions, verbose=False)


        fold_results.append({
            "fold": fold_num,
            "k": k,
            "RMSE": rmse,
            "MAE": mae,
            "Time": elapsed_time
        })

        for pred in predictions:
            all_predictions.append({
                "fold": fold_num,
                "userId": pred.uid,
                "movieId": pred.iid,
                "actual_rating": pred.r_ui,
                "predicted_rating": pred.est,
                "algorithm": "KNN",
                "k": k
            })

    results_df = pd.DataFrame(fold_results)
    predictions_df = pd.DataFrame(all_predictions)


    avg_rmse = results_df["RMSE"].mean()
    avg_mae = results_df["MAE"].mean()
    avg_time = results_df["Time"].mean()

    return results_df, predictions_df, avg_rmse, avg_mae, avg_time

In [33]:
#testing different k values 

#k values i want to test 
k_values = [4, 8, 10, 20, 40, 80, 100, 120]

#holds the results from the run with each k value 
summary_results = []

for k in k_values:

    fold_results, predictions, avg_rmse, avg_mae, avg_time = run_knn_cv(
        folds=folds,
        k=k,
        user_based=False,
        save_predictions=False
    )

    #save information from the runs with each k 
    summary_results.append({
        "k": k,
        "Average RMSE": avg_rmse,
        "Average MAE": avg_mae,
        "Average Time(sec)": avg_time
    })

knn_summary = pd.DataFrame(summary_results)
knn_summary = knn_summary.sort_values("Average RMSE")

knn_summary
#going to use k=80 because increasing the k after that doesn't significantly improve the RMSE or MAE after that and the time is similar 

,k,Average RMSE,Average MAE,Average Time(sec)
7,120,0.958611,0.744324,3.891445
6,100,0.961171,0.746734,3.780280
5,80,0.964464,0.749778,3.627304
4,40,0.977242,0.760655,3.262232
3,20,0.996517,0.776532,3.045998
2,10,1.025992,0.800367,2.901758
1,8,1.040930,0.811983,2.869782
0,4,1.106812,0.861850,2.949178


In [34]:
#create the final predictions csv 
final_fold_results, final_knn_predictions, final_rmse, final_mae, final_time = run_knn_cv(
    folds=folds,
    k=80,
    user_based=False,
)

In [35]:
final_knn_predictions.to_csv("knn_predictions_k80.csv", index=False)

knn_preds_relevant_columns = final_knn_predictions[["userId", "movieId", "predicted_rating"]]

#save it to the outputs folder 
knn_preds_relevant_columns.to_csv(
    os.path.join(REPO_ROOT, "outputs", f"knn_predictions_k80.csv"),
    index=False
)



Cleaning up matrix factorization

In [36]:
from surprise import SVD, accuracy 
#Defining matrix factorization with regularization 
#going to use surprise's svd but tweak some of the parameters 


SVD(
    n_factors=50,      #hidden dimensions for each movie/user
    n_epochs=20,       #number of times the model loops over the training data 
    lr_all=0.005,      #learning rate- how aggressive model adjusts after each mistake 
    reg_all=0.02,      #regularization- penalizes large values in user/movie vector and biases 
    biased=True        #takes into account biases 
)

#best number of factors/regularization combo was 100 factors adn 0.05 regularization 
def run_mf_reg_cv(folds, n_factors=100, reg_all=0.05, n_epochs=20, lr_all=0.005, save_predictions=False):
    fold_results = []
    all_predictions = []

    for fold_num, (trainset, testset) in enumerate(folds, start=1):

        model = SVD(
            n_factors=n_factors,
            n_epochs=n_epochs,
            lr_all=lr_all,
            reg_all=reg_all,
            biased=True,
            random_state=42
        )

        start_time = time.time()

        model.fit(trainset)
        predictions = model.test(testset)

        end_time = time.time()
        elapsed_time = end_time - start_time

        rmse = accuracy.rmse(predictions, verbose=False)
        mae = accuracy.mae(predictions, verbose=False)

        fold_results.append({
            "fold": fold_num,
            "n_factors": n_factors,
            "reg_all": reg_all,
            "n_epochs": n_epochs,
            "RMSE": rmse,
            "MAE": mae,
            "Time": elapsed_time
        })

        for pred in predictions:
            all_predictions.append({
                "userId": pred.uid,
                "movieId": pred.iid,
                "predicted_rating": pred.est
            })

    results_df = pd.DataFrame(fold_results)
    predictions_df = pd.DataFrame(all_predictions)

    avg_rmse = results_df["RMSE"].mean()
    avg_mae = results_df["MAE"].mean()
    avg_time = results_df["Time"].mean()

    if save_predictions:
        output_path = os.path.join(REPO_ROOT, "outputs", "mf_reg_predictions.csv")
        predictions_df.to_csv(output_path, index=False)
        print("Saved predictions to:", output_path)

    return results_df, predictions_df, avg_rmse, avg_mae, avg_time



In [37]:
#testing parameters for mf 
mf_results = []

#only going to test n factors and reg all parameters 
#reg_all -> smaller higher risk of over fitting, larger risk of under fitting 
for n_factors in [10, 20, 50, 80, 100]:
    for reg_all in [0.01, 0.02, 0.05, 0.07, 0.1]:

        fold_results, predictions, avg_rmse, avg_mae, avg_time = run_mf_reg_cv(
            folds=folds,
            n_factors=n_factors,
            reg_all=reg_all,
            n_epochs=20,
            lr_all=0.005,
        )

        mf_results.append({
            "n_factors": n_factors,
            "reg_all": reg_all,
            "Average RMSE": avg_rmse,
            "Average MAE": avg_mae,
            "Average Time": avg_time
        })

mf_summary = pd.DataFrame(mf_results)
mf_summary
#smallest average RMSE and MAE was with 100 factors and 0.05 regularization, RMSE(0.868899), MAE(0.667954) 

,n_factors,reg_all,Average RMSE,Average MAE,Average Time
0,10,0.01,0.869853,0.668065,0.180369
1,10,0.02,0.869359,0.667873,0.157637
2,10,0.05,0.869720,0.668556,0.142488
3,10,0.07,0.870193,0.669111,0.155010
4,10,0.10,0.870980,0.669978,0.153476
5,20,0.01,0.872464,0.670415,0.163793
6,20,0.02,0.870551,0.669046,0.175342
7,20,0.05,0.869866,0.668821,0.175865
8,20,0.07,0.870226,0.669262,0.176775
9,20,0.10,0.870979,0.670070,0.176148


In [27]:
final_mf_results, final_mf_predictions, mf_rmse, mf_mae, mf_time = run_mf_reg_cv(
    folds=folds,
    n_factors=100,
    reg_all=0.05,
    n_epochs=20,
    lr_all=0.005,
    save_predictions=True
)

final_mf_results

Saved predictions to: /Users/mauraschorr/Downloads/recommender-system/outputs/mf_reg_predictions.csv


,fold,n_factors,reg_all,n_epochs,RMSE,MAE,Time
0,1,100,0.05,20,0.879248,0.672489,0.457143
1,2,100,0.05,20,0.863341,0.665051,0.365590
2,3,100,0.05,20,0.859161,0.661923,0.426164
3,4,100,0.05,20,0.866935,0.666837,0.366298
4,5,100,0.05,20,0.875812,0.673469,0.365169
